In [ ]:
import fitz  # PyMuPDF
import re
import pandas as pd

In [ ]:
pdf_path = "MICO.pdf"

# --- Expresión regular robusta ---
pattern = re.compile(
    r"([A-Z0-9\-]+)\s+"             # Doc/Factura
    r"(\d+)\s+"                     # No.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Contabiliz.
    r"([\d.,]+-?)\s+"               # Importe (puede terminar en '-')
    r"([A-Z]{3})"                   # Moneda
)

rows = []

with fitz.open(pdf_path) as doc:
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        matches = pattern.findall(text)
        print(f"📄 Página {page_num} — {len(matches)} filas detectadas")
        for m in matches:
            rows.append(m)

# --- Crear DataFrame ---
df = pd.DataFrame(
    rows,
    columns=[
        "Doc/Factura",
        "No.Documento",
        "Fc.Documento",
        "Fc.Contabiliz.",
        "Importe",
        "Moneda",
    ],
)

# --- Limpieza de importes (siempre positivos) ---
def parse_importe(val):
    """
    Convierte valores tipo '27,024.63-' o '895.76-' en floats POSITIVOS (sin signo negativo).
    """
    if not isinstance(val, str):
        return None
    val = val.strip().replace("-", "")  # Quitamos el signo

    # Si hay ',' y '.', asumimos formato tipo '27,024.63' → quitamos ',' (miles)
    if "," in val and "." in val:
        val = val.replace(",", "")
    # Si hay solo ',', asumimos que ',' son decimales (formato europeo)
    elif "," in val and "." not in val:
        val = val.replace(",", ".")

    try:
        return float(val)
    except ValueError:
        return None

df["Importe"] = df["Importe"].apply(parse_importe)

print(f"\n✅ Total filas encontradas: {len(df)}")
display(df.head(15))